# API -> 데이터프레임 -> csv 저장
- https://www.data.go.kr/tcs/dss/selectApiDataDetailView.do?publicDataPk=15141809
- 가상환경 세팅 후에 실시해주세요. 본 폴더 내에 첨부파일 참고.
- 가상환경으로 변경은 **메뉴 > Kernel > Change Kernel**
- env 파일도 같이 올려놨으나, API키는 본인이 신청하여 받아 사용하는 편이 좋을 것 같습니다. 트래픽 제한이 있어서.
- 품목 코드에 주석처리하여 본인이 담당하는 것만 해주시면 됩니다.
- 담당은 다음과 같습니다. **양파 : 진성, 배추 : 용곤, 상추 : 동현, 사과 : 재성**
- 기간은 다음과 같습니다. **2018-01-01 ~ 2025-05-31**
- 데이터가 있는 것들은 확인했고, fail_log는 대체로 휴일이거나, 해당 시장에서는 해당 품목의 데이터가 없는 경우였습니다.
    * 만일 추가되는 오류 현상이 있다면 단톡에 공유바랍니다.
- 기간을 적게 잡아 테스트 (휴일피해서) 진행하고, 파일 확인 후, 목표 기간에 맞춰 진행해주세요. 
- 기간을 잘라서 진행해도 괜찮으나 최종 제출 파일은 기간 통합하여 올려주세요.

In [20]:
import os
import requests
import pandas as pd
import time
from tqdm import tqdm
from datetime import datetime, timedelta
import ssl
import warnings
from requests.packages.urllib3.exceptions import InsecureRequestWarning
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
# SSL 및 경고 설정
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context

# 디렉토리 준비

os.makedirs("data/fail_log", exist_ok=True)

# API 설정
API_KEY = os.getenv('DO_API_KEY')  # 환경 변수에서 키를 불러옵니다
BASE_URL = 'http://apis.data.go.kr/B552845/katSale/trades'

if not API_KEY:
    raise ValueError("환경변수 'DO_API_KEY'가 설정되지 않았습니다.")

# 도매시장 코드 불러오기
df_market = pd.read_csv('도매시장_코드.csv', encoding='cp949')

# 품목 코드 설정
ITEM_CODES = {
#    "양파": "1201",  # 진성
#     "배추": "1001",  # 용곤
#     "상추": "1005",  # 동현
#     "사과": "0601"   # 재성
#    "무" : '1101',
    "감자" : '0501',
#     "대파" : "1202",
#     "건고추" : "1207",
#     "마늘" : "1209",
#     "딸기" : '0804',
#     "방울토마토" : "0806",
#     "오이" : "0901",
#     "양배추" : "1004",
#     "고구마" : '0502',
#     '배' : '0602'
}

# 날짜 입력 (YYYY-MM-DD)  # 테스트 후에 일자 조정 => 
start_date = '2018-01-03'
end_date = '2019-12-31'
start_dt = datetime.strptime(start_date, '%Y-%m-%d')
end_dt = datetime.strptime(end_date, '%Y-%m-%d')
total_days = (end_dt - start_dt).days + 1

# 기타 설정
max_retries = 3
FAIL_LOG = []


# 품목별 반복
for item_name, code in tqdm(ITEM_CODES.items(), desc="전체 품목 진행"):
    LARGE = code[:2]
    MID = code[2:]
    data_list = []

    print(f"\n📦 {item_name} 수집 시작: {start_date} ~ {end_date}")
    print("📈 진행률: ", end='')

    current_dt = start_dt
    count = 0
    day_success_count = 0
    day_fail_count = 0

    while current_dt <= end_dt:
        # 주말(토요일=5, 일요일=6)이면 다음 날로 건너뜀
        if current_dt.weekday() >= 5:
            current_dt += timedelta(days=1)
            continue
            
        date_str = current_dt.strftime('%Y-%m-%d')  # API 포맷
        day_success_count = 0
        day_fail_count = 0
        for mcode, market_name in df_market.values:
            retry_count = 0
            market_success = False
            page_no = 1

            while retry_count < max_retries:
                try:
                    while True:
                        params = {
                            'serviceKey': API_KEY,
                            'pageNo': page_no,
                            'numOfRows': 100,
                            'cond[trd_clcln_ymd::EQ]': date_str,
                            'cond[whsl_mrkt_cd::EQ]': mcode,
                            'cond[gds_lclsf_cd::EQ]': LARGE,
                            'cond[gds_mclsf_cd::EQ]': MID
                        }

                        response = requests.get(BASE_URL, params=params, verify=False, timeout=10)
                        if response.status_code != 200:
                            fail_reason = f"HTTP {response.status_code}: {response.text[:100]}"
                            retry_count += 1
                            break

                        json_data = response.json()
                        header = json_data.get('response', {}).get('header', {})
                        body = json_data.get('response', {}).get('body', {})
                        items = body.get('items', {}).get('item', [])
                        total_count = int(body.get('totalCount', 0))

                        if isinstance(items, list) and items:
                            data_list.extend(items)
                            market_success = True
                            day_success_count += 1
                        elif isinstance(items, dict):  # 단일 객체일 경우
                            data_list.append(items)
                            market_success = True
                            day_success_count += 1
                        else:
                            break  # 데이터 없음

                        if page_no * 100 >= total_count:
                            break
                        else:
                            page_no += 1

                    break  # 내부 페이지 반복 성공 시 탈출

                except Exception as e:
                    retry_count += 1
                    fail_reason = f"Exception: {str(e)}"
                    time.sleep(0.3)

            if not market_success:
                FAIL_LOG.append({
                    "item": item_name,
                    "market": market_name,
                    "mcode": mcode,
                    "date": date_str,
                    "reason": fail_reason if 'fail_reason' in locals() else 'Unknown'
                })
                day_fail_count += 1

        # 진행률 출력
        count += 1
        if count % 10 == 0:
            percent = int((count / total_days) * 100)
            print(f"[누적{count}회({percent}%)]", end='', flush=True)
        else:
            print(f'|O{day_success_count}/X{day_fail_count}|', end='', flush=True)
            
        if count % 10 == 0 and data_list:
            os.makedirs(f"data/{item_name}", exist_ok=True)
            os.makedirs(f"data/{item_name}/backup", exist_ok=True)
            df_temp = pd.DataFrame(data_list)
            filename_temp = f"data/{item_name}/backup/도매시장_MID_{item_name}_{start_date.replace('-', '')}-{date_str.replace('-', '')}({market_name}).csv"
            df_temp.to_csv(filename_temp, encoding='cp949', index=False)
            df_fail_temp = pd.DataFrame(FAIL_LOG)
            df_fail_temp.to_csv(f'data/fail_log/유통공사_fail_log_중간저장_{count}.csv', index=False, encoding='cp949')
            print(f"💾", end='', flush=True)

        current_dt += timedelta(days=1)
        time.sleep(0.1)

    print(f"\n✅ {item_name} 완료: {len(data_list):,}건")

    if data_list:
        os.makedirs(f"data/{item_name}", exist_ok=True)
        df = pd.DataFrame(data_list)
        filename = f"data/{item_name}/유통공사_도매시장_{item_name}_{start_date.replace('-', '')}-{end_date.replace('-', '')}.csv"
        df.to_csv(filename, encoding='cp949', index=False)
        print(f"💾 저장됨: {filename}")
        time.sleep(0.3)
    else:
        print(f"⚠️ {item_name}: 수집된 데이터 없음")

    # 실패 로그 저장
    if FAIL_LOG:
        df_fail = pd.DataFrame(FAIL_LOG)
        df_fail.to_csv('data/fail_log/유통공사_fail_log.csv', index=False, encoding='cp949')
        print(f"\n❗ 실패 요청 {len(FAIL_LOG)}건 기록됨: data/유통공사_fail_log.csv")
        time.sleep(0.3)
    else:
        print("\n🎉 모든 수집 성공, 실패 없음!")


전체 품목 진행:   0%|                                                                           | 0/11 [00:00<?, ?it/s]


📦 무 수집 시작: 2018-01-03 ~ 2019-12-31
📈 진행률: |O30/X3||O31/X2||O30/X3||O30/X3||O30/X3||O28/X5||O31/X2||O28/X5||O31/X2|[누적10회(1%)]💾|O31/X2||O30/X3||O31/X2||O31/X2||O31/X2||O29/X4||O30/X3||O30/X3||O30/X3|[누적20회(2%)]💾|O29/X4||O29/X4||O31/X2||O31/X2||O29/X4||O31/X2||O31/X2||O31/X2||O32/X1|[누적30회(4%)]💾|O31/X2||O18/X15||O0/X33||O26/X7||O30/X3||O29/X4||O31/X2||O30/X3||O31/X2|[누적40회(5%)]💾

전체 품목 진행:   0%|                                                                           | 0/11 [02:23<?, ?it/s]


KeyboardInterrupt: 